# Chapter 7 — Ordered multi-conductor RLGC

Engineer course · source candidate · CONVERGING

# Chapter 7 — Ordered multi-conductor RLGC

This complete Chapter is one clean-kernel execution unit. Its four web
Lessons are reading views over the ordered source fragments. QMD is the
editable authority; `chapter.ipynb` is a generated zero-output transport
artifact.

## Lesson 1 — Declare ordered N=2 RLGC

### Keep conductor order and full matrices explicit

The ordered signal conductors are `readout` and `filter`. `ground` is
reference metadata, not a fifth electrical Pin or Port.

In [ ]:
from IPython.display import display

from scnsim import (
    CircuitDiagramSpec,
    CircuitPlan,
    CircuitRun,
    DirectSolveSpec,
    ParameterDefinitions,
    ParameterSet,
    ParameterSpace,
    ParameterSpec,
    RLGC,
    RLGCParameterSpec,
    SCNSimValidationError,
    SParameterTrace,
    Theme,
    components,
    units as u,
)

inputs = ParameterDefinitions(id="coupled_line_design")
baseline_rlgc = RLGC(
    conductors=("readout", "filter"),
    reference_conductor="ground",
    resistance_per_length=[[0.18, 0.0], [0.0, 0.22]] * u.ohm / u.m,
    inductance_per_length=[[420.0, 75.0], [75.0, 395.0]] * u.nH / u.m,
    conductance_per_length=[[0.0, 0.0], [0.0, 0.0]] * u.S / u.m,
    capacitance_per_length=[[175.0, -22.0], [-22.0, 168.0]] * u.pF / u.m,
)
line_length = inputs.parameter(
    id="line_length",
    baseline=1.6 * u.mm,
    spec=ParameterSpec(unit=u.mm),
)
line_rlgc = inputs.parameter(
    id="line_rlgc",
    baseline=baseline_rlgc,
    spec=RLGCParameterSpec(
        conductors=("readout", "filter"), reference_conductor="ground"
    ),
)

R is diagonal, L is symmetric with +75 nH/m mutual entries, G is exactly
zero, and C is symmetric with -22 pF/m off-diagonals. A whole RLGC value
is one structured parameter; its individual matrix entries are not sweep
axes.

In [ ]:
plan = CircuitPlan(id="ordered_n2_line")
line = plan.add(
    components.transmission_line(
        id="coupled",
        length=line_length,
        rlgc=line_rlgc,
        n_sections=8,
    )
)

`n_sections=8` is fixed model structure. The finite-pi expansion retains
the full coupled matrices; it does not split this body into two scalar
lines.

## Lesson 2 — Bind four ordered signal Ports

### Use the complete occurrence once

The body’s four public signal Pins are ordered by conductor and
head/tail.

In [ ]:
readout_head_pin = line.pin("head", conductor="readout")
readout_tail_pin = line.pin("tail", conductor="readout")
filter_head_pin = line.pin("head", conductor="filter")
filter_tail_pin = line.pin("tail", conductor="filter")

signal_buses = {
    name: plan.bus(id=name)
    for name in ("readout_head", "readout_tail", "filter_head", "filter_tail")
}
plan.series(
    id="readout_conductor",
    start=signal_buses["readout_head"],
    elements=(line.between(readout_head_pin, readout_tail_pin),),
    end=signal_buses["readout_tail"],
)
plan.link(
    id="filter_head_binding",
    endpoints=(signal_buses["filter_head"], filter_head_pin),
)
plan.link(
    id="filter_tail_binding",
    endpoints=(signal_buses["filter_tail"], filter_tail_pin),
)

`.between()` supplies one ordered endpoint pair for the complete
occurrence; it does not extract the readout conductor or clone the line.
The explicit Links bind the remaining signal Pins.

In [ ]:
port_ids = ("readout_head", "readout_tail", "filter_head", "filter_tail")
ports = {
    name: plan.add_port(
        id=name,
        at=signal_buses[name],
        role="terminated",
        reference_impedance=50.0 * u.ohm,
    )
    for name in port_ids
}

There are exactly four 50 ohm signal Ports.
`reference_conductor="ground"` does not create another Port or public
Pin.

## Lesson 3 — Separate authoring and finite-pi views

### Review the declared body

In [ ]:
authoring = None
try:
    authoring = plan.render_schematic(
        CircuitDiagramSpec(
            representation="authoring",
            theme=Theme.AUTO,
            show_parameter_values=True,
        )
    )
except SCNSimValidationError as error:
    if error.stage != "schematic_layout":
        raise
    display({"authoring layout unavailable": str(error), "stage": error.stage})
if authoring is not None:
    display(authoring.show())

In [ ]:
if authoring is not None:
    authoring.audit.show()

### Request the compiled presentation independently

In [ ]:
compiled = None
try:
    compiled = plan.render_schematic(
        CircuitDiagramSpec(
            representation="compiled",
            theme=Theme.AUTO,
            show_parameter_values=True,
        )
    )
except SCNSimValidationError as error:
    if error.stage != "schematic_layout":
        raise
    display({"compiled layout unavailable": str(error), "stage": error.stage})
if compiled is not None:
    display(compiled.show())

When available, the compiled diagram expands eight finite pi sections
and its audit retains conductor order, full matrices, and exact-zero
evidence. A typed layout failure does not change the authored Plan or
numerical model.

## Lesson 4 — Evaluate three complete parameter points

### Change length or the whole capacitance matrix

The second point changes only length to 1.8 mm. The third scales the
complete C matrix by the named teaching factor 170/175; R, L, and G
remain byte-for-value the baseline declaration and `n_sections` remains
8.

In [ ]:
scaled_rlgc = RLGC(
    conductors=("readout", "filter"),
    reference_conductor="ground",
    resistance_per_length=baseline_rlgc.resistance_per_length,
    inductance_per_length=baseline_rlgc.inductance_per_length,
    conductance_per_length=baseline_rlgc.conductance_per_length,
    capacitance_per_length=(170.0 / 175.0)
    * baseline_rlgc.capacitance_per_length,
)
parameter_points = ParameterSpace.points(
    (
        ParameterSet(),
        ParameterSet({line_length: 1.8 * u.mm}),
        ParameterSet({line_rlgc: scaled_rlgc}),
    )
)
point_labels = ("baseline", "length_1p8_mm", "capacitance_times_170_over_175")

This is a list of three complete points, not Q2D, an element scan, a
matrix-entry axis, or an interpolation contract.

### Name the complete four-Port channel inventory

The Direct Result contains the complete 4×4 S matrix. Sixteen named
trace projections expose every output-from-input channel without
launching another solve.

In [ ]:
channel_traces = tuple(
    SParameterTrace(
        id=f"s_{output}_from_{input_}",
        input_port=input_,
        input_mode=(),
        output_port=output,
        output_mode=(),
    )
    for output in port_ids
    for input_ in port_ids
)
run = CircuitRun(plan=plan, workspace="workspaces/engineer-chapter-07")
direct_spec = DirectSolveSpec(
    frequencies=[5.5, 6.0, 6.5] * u.GHz,
    traces=channel_traces,
)
run.explain(run.original, direct_spec, parameters=ParameterSet()).show()

The three frequencies are declared samples. No curve between them is
used to claim a continuous resonance or root.

In [ ]:
direct_points = run.solve(
    run.original,
    direct_spec,
    parameters=parameter_points,
)
direct_points.show()

In [ ]:
for label, point in zip(point_labels, direct_points.points, strict=True):
    if not point.succeeded:
        display({label: point.failure})
        continue
    display({"point": label, "matrix coordinates": point.result.s.view.coordinates})
    display(point.result.traces["s_readout_head_from_readout_head"].show(magnitude="db"))

The plotted `[0,0]`-position example is explicitly named
`s_readout_head_from_readout_head`: readout-head reflection. It is not
described as the complete matrix. The generated table and CSV inventory
all sixteen named channels for every point.